# MS2 Training — Arabic NLP QA RAG
**Before running:** Runtime → Change runtime type → **T4 GPU**

Run all cells top to bottom. Takes ~40 min total.

In [ ]:
# 1. Clone repo + install deps
!git clone https://github.com/seif495/arabic-nlp-qa-rag
%cd arabic-nlp-qa-rag
!pip install sentencepiece tensorflow -q

In [ ]:
# 2. Verify GPU
import tensorflow as tf
print('TF:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'No GPU detected — change runtime type to T4 GPU and restart'

In [ ]:
# 3. Run MS1 pipeline — cleans transcripts, builds ms1_dataset_processed_v001.jsonl
!python -m src.cli.ms1 run-all

In [ ]:
# 4. Verify MS1 output
import json, os
path = 'data/processed/ms1/ms1_dataset_processed_v001.jsonl'
with open(path, encoding='utf-8') as f:
    records = [json.loads(l) for l in f]
print(f'{len(records)} records  |  splits:', {r['split'] for r in records})

In [ ]:
# 5. Train tokenizer + build pipeline cache (both model A and B schemas)
!python -m src.cli.ms2 prep-data --target-model a
!python -m src.cli.ms2 prep-data --target-model b

In [ ]:
# 6. Load cached examples into memory
import json
from pathlib import Path
from src.common.paths import resolve_ms2_paths

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f]

def find_cache(target_model, split):
    paths = resolve_ms2_paths(split=f'{split}_{target_model}')
    return paths.tfrecord_shard_dir / 'examples.jsonl'

cache_a_train = load_jsonl(find_cache('a', 'train'))
cache_a_dev   = load_jsonl(find_cache('a', 'dev'))
cache_b_train = load_jsonl(find_cache('b', 'train'))
cache_b_dev   = load_jsonl(find_cache('b', 'dev'))
print(f'Model A  train={len(cache_a_train)}  dev={len(cache_a_dev)}')
print(f'Model B  train={len(cache_b_train)}  dev={len(cache_b_dev)}')

In [ ]:
# 7. Build tf.data.Dataset helpers
import numpy as np
import tensorflow as tf

BATCH_SIZE = 8

def examples_to_dataset_a(examples, shuffle=False):
    q  = np.array([e['question_ids']      for e in examples], dtype=np.int32)
    c  = np.array([e['context_ids']       for e in examples], dtype=np.int32)
    ei = np.array([e['encoder_input_ids'] for e in examples], dtype=np.int32)
    cm = np.array([e['char_matrix']       for e in examples], dtype=np.int32)
    di = np.array([e['decoder_input_ids'] for e in examples], dtype=np.int32)
    dt = np.array([e['decoder_target_ids']for e in examples], dtype=np.int32)
    lm = np.array([e['loss_mask']         for e in examples], dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices(
        {'q': q, 'c': c, 'ei': ei, 'cm': cm, 'di': di, 'dt': dt, 'lm': lm}
    )
    if shuffle:
        ds = ds.shuffle(min(len(examples), 1024), seed=42)
    return ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(2)

def examples_to_dataset_b(examples, shuffle=False):
    ei = np.array([e['encoder_input_ids'] for e in examples], dtype=np.int32)
    di = np.array([e['decoder_input_ids'] for e in examples], dtype=np.int32)
    dt = np.array([e['decoder_target_ids']for e in examples], dtype=np.int32)
    lm = np.array([e['loss_mask']         for e in examples], dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices(
        {'ei': ei, 'di': di, 'dt': dt, 'lm': lm}
    )
    if shuffle:
        ds = ds.shuffle(min(len(examples), 1024), seed=42)
    return ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(2)

ds_a_train = examples_to_dataset_a(cache_a_train, shuffle=True)
ds_a_dev   = examples_to_dataset_a(cache_a_dev)
ds_b_train = examples_to_dataset_b(cache_b_train, shuffle=True)
ds_b_dev   = examples_to_dataset_b(cache_b_dev)
print('Datasets ready')

In [ ]:
# 8. Build trainer wrappers for Model A and Model B
import time
import tensorflow as tf
from src.ms2.models.rnn.model_a import ModelA
from src.ms2.models.transformer.model_b import ModelB
from src.ms2.training.optimizer import build_adamw
from src.ms2.training.schedules import CosineWithWarmup, Noam
from src.ms2.training.loss import label_smoothed_cross_entropy
from src.ms2.metrics.scoring import token_f1, exact_match, char_edit_distance_normalized, bleu1
from src.ms2.metrics.normalize import arabic_post_normalize
from src.ms2.data.tokenizer import ensure_default_tokenizer, EOS_ID, PAD_ID

TOTAL_STEPS = 3000   # conservative estimate for 15-min budget
SMOOTHING   = 0.1
CLIP_NORM   = 1.0
SEED        = 42

tokenizer = ensure_default_tokenizer()

def decode_ids(ids):
    """Decode token ids to string, stopping at EOS."""
    clipped = []
    for i in ids:
        if int(i) == EOS_ID:
            break
        if int(i) != PAD_ID:
            clipped.append(int(i))
    return tokenizer.decode(clipped)

def compute_metrics(pred_ids_batch, target_ids_batch):
    preds = [decode_ids(p) for p in pred_ids_batch]
    refs  = [decode_ids(t) for t in target_ids_batch]
    em  = float(np.mean([exact_match(p, r) for p, r in zip(preds, refs)]))
    f1  = float(np.mean([token_f1(p, r)   for p, r in zip(preds, refs)]))
    ced = float(np.mean([char_edit_distance_normalized(p, r) for p, r in zip(preds, refs)]))
    b1  = float(np.mean([bleu1(p, r)      for p, r in zip(preds, refs)]))
    return {'em': em, 'token_f1': f1, 'char_edit_distance': ced, 'bleu1': b1}


class ModelATrainer:
    def __init__(self):
        tf.keras.utils.set_random_seed(SEED)
        self.model = ModelA()
        schedule = CosineWithWarmup(total_steps=TOTAL_STEPS)
        self.opt_bundle = build_adamw(learning_rate=schedule, gradient_clip_norm=CLIP_NORM)
        self.optimizer = self.opt_bundle.optimizer
        self.optimizer_bundle = self.opt_bundle
        self.parameter_count = 0
        self.peak_gpu_memory_mb = 0.0
        self.mean_inference_time_ms_per_example = 0.0

    @tf.function
    def _step(self, batch):
        q, c, ei, cm = batch['q'], batch['c'], batch['ei'], batch['cm']
        di, dt, lm   = batch['di'], batch['dt'], batch['lm']
        with tf.GradientTape() as tape:
            out    = self.model((q, c, ei, cm, di), training=True)
            logits = tf.cast(out['logits'], tf.float32)
            loss   = label_smoothed_cross_entropy(logits, dt, lm, smoothing=SMOOTHING)
        grads = tape.gradient(loss, self.model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, CLIP_NORM)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

    def train_step(self, batch):
        return self._step(batch)

    def evaluate(self, dataset_dev):
        all_preds, all_targets = [], []
        for batch in dataset_dev:
            q, c, ei, cm = batch['q'], batch['c'], batch['ei'], batch['cm']
            di, dt = batch['di'], batch['dt']
            out     = self.model((q, c, ei, cm, di), training=False)
            preds   = tf.argmax(tf.cast(out['logits'], tf.float32), axis=-1).numpy()
            all_preds.extend(preds)
            all_targets.extend(dt.numpy())
        if not self.parameter_count:
            self.parameter_count = int(self.model.count_params())
        return compute_metrics(all_preds, all_targets)


class ModelBTrainer:
    def __init__(self):
        tf.keras.utils.set_random_seed(SEED)
        self.model = ModelB()
        schedule = Noam(d_model=192, warmup_steps=1000)
        self.opt_bundle = build_adamw(learning_rate=schedule, gradient_clip_norm=CLIP_NORM)
        self.optimizer = self.opt_bundle.optimizer
        self.optimizer_bundle = self.opt_bundle
        self.parameter_count = 0
        self.peak_gpu_memory_mb = 0.0
        self.mean_inference_time_ms_per_example = 0.0

    @tf.function
    def _step(self, batch):
        ei, di, dt, lm = batch['ei'], batch['di'], batch['dt'], batch['lm']
        with tf.GradientTape() as tape:
            out    = self.model((ei, di), training=True)
            logits = tf.cast(out['logits'], tf.float32)
            loss   = label_smoothed_cross_entropy(logits, dt, lm, smoothing=SMOOTHING)
        grads = tape.gradient(loss, self.model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, CLIP_NORM)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss

    def train_step(self, batch):
        return self._step(batch)

    def evaluate(self, dataset_dev):
        all_preds, all_targets = [], []
        for batch in dataset_dev:
            ei, di, dt = batch['ei'], batch['di'], batch['dt']
            out   = self.model((ei, di), training=False)
            preds = tf.argmax(tf.cast(out['logits'], tf.float32), axis=-1).numpy()
            all_preds.extend(preds)
            all_targets.extend(dt.numpy())
        if not self.parameter_count:
            self.parameter_count = int(self.model.count_params())
        return compute_metrics(all_preds, all_targets)

print('Trainer classes defined')

In [ ]:
# 9. Train Model A (RNN + gated merge + FiLM) — ~15 min
from src.ms2.training.loop import train_one_run
from src.ms2.schemas import RunConfig

run_config_a = RunConfig(
    model_id='A', seed=SEED,
    wall_clock_budget_minutes=15,
    lr_schedule='cosine_with_warmup',
)

trainer_a = ModelATrainer()
print('Model A params:', trainer_a.model.count_params())

summary_a = train_one_run(
    model=trainer_a,
    dataset_train=ds_a_train.repeat(),  # repeat so budget controls stopping
    dataset_dev=ds_a_dev,
    run_config=run_config_a,
)
print('\nModel A training done')
print(f'  dev EM={summary_a.dev_em:.4f}  F1={summary_a.dev_token_f1:.4f}  wall_clock={summary_a.train_wall_clock_minutes:.1f} min')

In [ ]:
# 10. Train Model B (Transformer + RoPE) — ~15 min
run_config_b = RunConfig(
    model_id='B', seed=SEED,
    wall_clock_budget_minutes=15,
    lr_schedule='noam',
)

trainer_b = ModelBTrainer()
print('Model B params:', trainer_b.model.count_params())

summary_b = train_one_run(
    model=trainer_b,
    dataset_train=ds_b_train.repeat(),
    dataset_dev=ds_b_dev,
    run_config=run_config_b,
)
print('\nModel B training done')
print(f'  dev EM={summary_b.dev_em:.4f}  F1={summary_b.dev_token_f1:.4f}  wall_clock={summary_b.train_wall_clock_minutes:.1f} min')

In [ ]:
# 11. Print headline results and update the report table
from pathlib import Path
import json

headline = f"""# MS2 Headline Table

| Metric | Model A (RNN) | Model B (Transformer) |
| --- | --- | --- |
| Exact Match (dev, seed 42) | {summary_a.dev_em:.4f} | {summary_b.dev_em:.4f} |
| Token-F1 (dev, seed 42) | {summary_a.dev_token_f1:.4f} | {summary_b.dev_token_f1:.4f} |
| Char edit distance (dev) | {summary_a.dev_char_edit_distance:.4f} | {summary_b.dev_char_edit_distance:.4f} |
| BLEU-1 (dev) | {summary_a.dev_bleu1:.4f} | {summary_b.dev_bleu1:.4f} |
| Parameter count | {trainer_a.parameter_count:,} | {trainer_b.parameter_count:,} |
| Training wall-clock | {summary_a.train_wall_clock_minutes:.1f} min (budget 15 min, 1 seed) | {summary_b.train_wall_clock_minutes:.1f} min (budget 15 min, 1 seed) |

_Single seed (42), 15-min compute budget per model. Full 3-seed × 75-min runs omitted due to resource constraints._
"""

Path('docs/reports/ms2_headline_table.md').write_text(headline, encoding='utf-8')
print(headline)

In [ ]:
# 12. Download results
import shutil
shutil.make_archive('ms2_results', 'zip', '.', 'experiments/ms2')
shutil.make_archive('ms2_reports', 'zip', '.', 'docs/reports')

from google.colab import files
files.download('ms2_results.zip')
files.download('ms2_reports.zip')